In [1]:
# --- Instrument profiles (minimal schema) ---
# MIDI pitch numbers (C4 = 60)
INSTRUMENT_PROFILES = {
    # Woodwind
    "FLUTE": {
        "name": "Flute",
        "category": "woodwind",
        "min_pitch": 60,
        "max_pitch": 96,
        "preferred_low": 67,
        "preferred_high": 88,
        "max_polyphony": 1
    },
    "CLARINET": {
        "name": "Clarinet",
        "category": "woodwind",
        "min_pitch": 50,
        "max_pitch": 91,
        "preferred_low": 60,
        "preferred_high": 84,
        "max_polyphony": 1
    },

    # Vocal
    "TENOR_VOICE": {
        "name": "Tenor Voice",
        "category": "vocal",
        "min_pitch": 48,   # C3
        "max_pitch": 69,   # A4
        "preferred_low": 52,  # E3
        "preferred_high": 64, # E4
        "max_polyphony": 1
    },

    # Brasswind
    "TRUMPET": {
        "name": "Trumpet",
        "category": "brasswind",
        "min_pitch": 55,   # G3
        "max_pitch": 82,   # A#5
        "preferred_low": 60,  # C4
        "preferred_high": 76, # E5
        "max_polyphony": 1
    },

    # Strings
    "VIOLIN": {
        "name": "Violin",
        "category": "strings",
        "min_pitch": 55,    # G3
        "max_pitch": 103,   # G7
        "preferred_low": 62,  # D4
        "preferred_high": 96, # C7
        "max_polyphony": 1
    },

    # Percussion
    "MARIMBA": {
        "name": "Marimba",
        "category": "percussion",
        "min_pitch": 45,    # A2
        "max_pitch": 93,    # A6
        "preferred_low": 52,  # E3
        "preferred_high": 84, # C6
        "max_polyphony": 4   # mallet instruments can sound multiple notes
    },
}

# Convenience handles
FLUTE = INSTRUMENT_PROFILES["FLUTE"]
CLARINET = INSTRUMENT_PROFILES["CLARINET"]
TENOR_VOICE = INSTRUMENT_PROFILES["TENOR_VOICE"]
TRUMPET = INSTRUMENT_PROFILES["TRUMPET"]
VIOLIN = INSTRUMENT_PROFILES["VIOLIN"]
MARIMBA = INSTRUMENT_PROFILES["MARIMBA"]

## Core functions
Hard constraints: pitch range + polyphony. Soft metric: tessitura violations.


In [2]:
# Constraint functions
# Each note is represented as (pitch, start_time, duration)

from typing import List, Tuple, Dict, Any, Optional
Note = Tuple[int, float, float]

def out_of_range(notes: List[Note], profile: Dict[str, Any]) -> bool:
    return any(
        (pitch < profile["min_pitch"]) or (pitch > profile["max_pitch"])
        for pitch, _, _ in notes
    )

def max_simultaneous_notes(notes: List[Note]) -> int:
    events = []
    for _, start, dur in notes:
        events.append((start, +1))
        events.append((start + dur, -1))
    events.sort()
    current = 0
    max_poly = 0
    for _, delta in events:
        current += delta
        max_poly = max(max_poly, current)
    return max_poly

def tessitura_violations(notes: List[Note], profile: Dict[str, Any]) -> int:
    return sum(
        (pitch < profile["preferred_low"]) or (pitch > profile["preferred_high"])
        for pitch, _, _ in notes
    )

def shift_notes(notes: List[Note], semitones: int) -> List[Note]:
    return [(pitch + semitones, start, dur) for pitch, start, dur in notes]

In [ ]:
## Correction logic
We compare a simple baseline to a constraint-aware method that selects among feasible octave shifts by minimizing tessitura violations.


In [3]:
# Simple vs advanced pitch correction

def simple_pitch_correction(notes: List[Note], profile: Dict[str, Any]) -> Dict[str, Any]:
    """
    Simple baseline:
    - Only tries to make notes fit the absolute range via octave shifts.
    - Ignores polyphony and tessitura comfort.
    """
    if not out_of_range(notes, profile):
        return {
            "status": "ACCEPTED",
            "reason": "Baseline: already within range",
            "notes": notes,
            "shift": 0,
            "soft": {"tessitura_violations": tessitura_violations(notes, profile)}
        }

    for octave in range(-3, 4):
        shifted = shift_notes(notes, octave * 12)
        if not out_of_range(shifted, profile):
            return {
                "status": "CORRECTED",
                "reason": f"Baseline: first-fit octave shift {octave*12} semitones to satisfy range",
                "notes": shifted,
                "shift": octave * 12,
                "soft": {"tessitura_violations": tessitura_violations(shifted, profile)}
            }

    return {
        "status": "REJECTED",
        "reason": "Baseline: no octave shift can satisfy range",
        "notes": None,
        "shift": None,
        "soft": {"tessitura_violations": tessitura_violations(notes, profile)}
    }


def find_best_feasible_transposition(notes: List[Note], profile: Dict[str, Any]) -> Tuple[Optional[List[Note]], Optional[int], Optional[int]]:
    """
    Advanced correction:
    - Enumerate all octave shifts that satisfy HARD constraints: range + polyphony.
    - Choose the candidate that minimizes tessitura violations (SOFT metric),
      tie-break by smallest absolute shift.
    Returns: (best_notes, best_shift, best_tessitura_violations)
    """
    candidates = []
    for octave in range(-3, 4):
        shift = octave * 12
        shifted = shift_notes(notes, shift)
        if out_of_range(shifted, profile):
            continue
        if max_simultaneous_notes(shifted) > profile["max_polyphony"]:
            continue
        tv = tessitura_violations(shifted, profile)
        candidates.append((tv, abs(shift), shift, shifted))

    if not candidates:
        return None, None, None

    candidates.sort(key=lambda x: (x[0], x[1]))
    tv, _, shift, best = candidates[0]
    return best, shift, tv


def constraint_aware_correction(notes: List[Note], profile: Dict[str, Any]) -> Dict[str, Any]:
    """
    Constraint-aware method:
    - Hard constraint 1: polyphony (reject if violated, since we don't rewrite texture yet)
    - Hard constraint 2: range (correctable via octave shifts)
    - Soft signal: tessitura violations (reported, and used for selecting best shift)
    """
    poly = max_simultaneous_notes(notes)
    if poly > profile["max_polyphony"]:
        return {
            "status": "REJECTED",
            "reason": f"Hard constraint: polyphony {poly} exceeds limit {profile['max_polyphony']}",
            "notes": None,
            "shift": None,
            "soft": {"tessitura_violations": tessitura_violations(notes, profile)}
        }

    if not out_of_range(notes, profile):
        tv = tessitura_violations(notes, profile)
        reason = "Hard constraints satisfied"
        if tv > 0:
            reason += f"; soft warning: {tv} tessitura violations"
        return {
            "status": "ACCEPTED",
            "reason": reason,
            "notes": notes,
            "shift": 0,
            "soft": {"tessitura_violations": tv}
        }

    best, shift, tv = find_best_feasible_transposition(notes, profile)
    if best is None:
        return {
            "status": "REJECTED",
            "reason": "Hard constraint: no feasible octave shift satisfies range (and polyphony)",
            "notes": None,
            "shift": None,
            "soft": {"tessitura_violations": tessitura_violations(notes, profile)}
        }

    reason = f"Corrected: chose octave shift {shift} semitones (min tessitura violations)"
    if tv and tv > 0:
        reason += f"; soft warning: {tv} tessitura violations"
    return {
        "status": "CORRECTED",
        "reason": reason,
        "notes": best,
        "shift": shift,
        "soft": {"tessitura_violations": tv}
    }

## Quick test cases


In [4]:
# Flute tests
melody_out_of_range = [(48, 0.0, 0.5), (50, 0.5, 0.5), (52, 1.0, 0.5)]
chord = [(72, 0.0, 1.0), (76, 0.0, 1.0), (79, 0.0, 1.0)]
valid_melody = [(72, 0.0, 0.5), (74, 0.5, 0.5), (76, 1.0, 0.5)]
boundary_melody = [(FLUTE["min_pitch"], 0.0, 0.5), (FLUTE["max_pitch"], 0.5, 0.5)]
unfixable_melody = [(40, 0.0, 0.5), (100, 0.5, 0.5)]

clarinet_valid = [(62, 0.0, 0.5), (65, 0.5, 0.5), (69, 1.0, 0.5)]
clarinet_out_of_range = [(45, 0.0, 0.5), (47, 0.5, 0.5), (49, 1.0, 0.5)]

marimba_chord = [(60, 0.0, 1.0), (64, 0.0, 1.0), (67, 0.0, 1.0)]

print("Constraint-aware (Flute chord):", constraint_aware_correction(chord, FLUTE))
print("Baseline (Flute chord):", simple_pitch_correction(chord, FLUTE))
print("Constraint-aware (Marimba chord):", constraint_aware_correction(marimba_chord, MARIMBA))

Constraint-aware (Flute chord): {'status': 'REJECTED', 'reason': 'Hard constraint: polyphony 3 exceeds limit 1', 'notes': None, 'shift': None, 'soft': {'tessitura_violations': 0}}
Baseline (Flute chord): {'status': 'ACCEPTED', 'reason': 'Baseline: already within range', 'notes': [(72, 0.0, 1.0), (76, 0.0, 1.0), (79, 0.0, 1.0)], 'shift': 0, 'soft': {'tessitura_violations': 0}}
Constraint-aware (Marimba chord): {'status': 'ACCEPTED', 'reason': 'Hard constraints satisfied', 'notes': [(60, 0.0, 1.0), (64, 0.0, 1.0), (67, 0.0, 1.0)], 'shift': 0, 'soft': {'tessitura_violations': 0}}


## Evaluation harness
Runs the same clips across instruments and methods and returns a results table.


In [5]:
import pandas as pd

METHODS = {
    "simple_pitch": simple_pitch_correction,
    "constraint_aware": constraint_aware_correction,
}

EVAL_CLIPS = {
    "flute_out_of_range": melody_out_of_range,
    "flute_chord": chord,
    "flute_valid": valid_melody,
    "flute_boundary": boundary_melody,
    "flute_unfixable": unfixable_melody,
    "clarinet_valid": clarinet_valid,
    "clarinet_out_of_range": clarinet_out_of_range,
    "marimba_chord": marimba_chord,
}

EVAL_INSTRUMENTS = {
    "FLUTE": FLUTE,
    "CLARINET": CLARINET,
    "TENOR_VOICE": TENOR_VOICE,
    "TRUMPET": TRUMPET,
    "VIOLIN": VIOLIN,
    "MARIMBA": MARIMBA,
}

def run_harness(clips, instruments, methods) -> pd.DataFrame:
    rows = []
    for clip_name, notes in clips.items():
        for inst_key, profile in instruments.items():
            for method_name, fn in methods.items():
                res = fn(notes, profile)
                playable = (res["notes"] is not None)
                rows.append({
                    "clip": clip_name,
                    "instrument": inst_key,
                    "category": profile.get("category", ""),
                    "method": method_name,
                    "status": res.get("status"),
                    "playable": playable,
                    "shift": res.get("shift"),
                    "tessitura_violations": (res.get("soft", {}) or {}).get("tessitura_violations"),
                    "reason": res.get("reason"),
                })
    return pd.DataFrame(rows)

df_results = run_harness(EVAL_CLIPS, EVAL_INSTRUMENTS, METHODS)

df_results.sort_values(["clip", "instrument", "method"]).head(30)

,clip,instrument,category,method,status,playable,shift,tessitura_violations,reason
75,clarinet_out_of_range,CLARINET,woodwind,constraint_aware,CORRECTED,True,24.0,0,Corrected: chose octave shift 24 semitones (mi...
74,clarinet_out_of_range,CLARINET,woodwind,simple_pitch,CORRECTED,True,12.0,2,Baseline: first-fit octave shift 12 semitones ...
73,clarinet_out_of_range,FLUTE,woodwind,constraint_aware,CORRECTED,True,24.0,0,Corrected: chose octave shift 24 semitones (mi...
72,clarinet_out_of_range,FLUTE,woodwind,simple_pitch,CORRECTED,True,24.0,0,Baseline: first-fit octave shift 24 semitones ...
83,clarinet_out_of_range,MARIMBA,percussion,constraint_aware,ACCEPTED,True,0.0,3,Hard constraints satisfied; soft warning: 3 te...
82,clarinet_out_of_range,MARIMBA,percussion,simple_pitch,ACCEPTED,True,0.0,3,Baseline: already within range
77,clarinet_out_of_range,TENOR_VOICE,vocal,constraint_aware,CORRECTED,True,12.0,0,Corrected: chose octave shift 12 semitones (mi...
76,clarinet_out_of_range,TENOR_VOICE,vocal,simple_pitch,CORRECTED,True,12.0,0,Baseline: first-fit octave shift 12 semitones ...
79,clarinet_out_of_range,TRUMPET,brasswind,constraint_aware,CORRECTED,True,24.0,0,Corrected: chose octave shift 24 semitones (mi...
78,clarinet_out_of_range,TRUMPET,brasswind,simple_pitch,CORRECTED,True,12.0,2,Baseline: first-fit octave shift 12 semitones ...


## Summary
Quick aggregate views for feasibility rate and tessitura comfort.


In [6]:
feas_rate = (
    df_results.groupby(["instrument", "method"])["playable"]
    .mean()
    .reset_index()
    .rename(columns={"playable": "feasible_rate"})
)

avg_tess = (
    df_results[df_results["playable"]]
    .groupby(["instrument", "method"])["tessitura_violations"]
    .mean()
    .reset_index()
    .rename(columns={"tessitura_violations": "avg_tessitura_violations"})
)

feas_rate.merge(avg_tess, on=["instrument", "method"], how="left").sort_values(["instrument", "method"])

,instrument,method,feasible_rate,avg_tessitura_violations
0,CLARINET,constraint_aware,0.500,0.000000
1,CLARINET,simple_pitch,0.750,0.333333
2,FLUTE,constraint_aware,0.625,0.800000
3,FLUTE,simple_pitch,0.875,1.285714
4,MARIMBA,constraint_aware,0.875,0.857143
5,MARIMBA,simple_pitch,0.875,0.857143
6,TENOR_VOICE,constraint_aware,0.500,1.000000
7,TENOR_VOICE,simple_pitch,0.750,1.333333
8,TRUMPET,constraint_aware,0.500,0.000000
9,TRUMPET,simple_pitch,0.750,0.500000
